# Special Days → OBS lakehouse (`obs://lakehouse-dev/special_events`)

Run this on the THY Spark nonprod cluster (JupyterHub, XS) **after `git pull`**.
It scrapes the next 12 months of special dates, enriches them (heuristic scorer,
no LLM), and writes two **Parquet datasets** (path-only — no Hive metastore) under
`obs://lakehouse-dev/special_events`:

| Path | Grain |
| --- | --- |
| `.../special_days_raw` | one row per special date (span grain) |
| `.../special_days_features` | one row per `(event_date, country, airport)` |

**Credentials:** put the OBS service-account keys in a `.env` at the repo root
(git-ignored — never commit them):

```
OBS_ENDPOINT=bigdata-dev.obs
OBS_ACCESS_KEY=<AK>
OBS_SECRET_KEY=<SK>
```

Leave them unset if the cluster session already has OBS access. The OBSA connector
jar (`hadoop-huaweicloud`) must be on the Spark classpath; if it isn't, launch the
kernel/session with `--jars /path/to/hadoop-huaweicloud.jar`.

In [ ]:
# Put the repo root on sys.path so `import special_days` works from the notebook.
import os, sys
cur = os.getcwd()
while cur != os.path.dirname(cur):
    if os.path.isdir(os.path.join(cur, 'special_days')):
        if cur not in sys.path:
            sys.path.insert(0, cur)
        break
    cur = os.path.dirname(cur)
print('repo root:', cur)

In [ ]:
# A Spark session may already be provided as `spark`; getOrCreate() reuses it.
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('special-days-lakehouse').getOrCreate()
spark

In [ ]:
# Point Spark at OBS using the service-account keys from .env (no-op if unset).
from special_days.config import load_dotenv, get_obs_endpoint, get_obs_access_key, get_obs_secret_key
from special_days.sinks import lakehouse

load_dotenv()
applied = lakehouse.configure_obs(
    spark,
    endpoint=get_obs_endpoint(),
    access_key=get_obs_access_key(),
    secret_key=get_obs_secret_key(),
)
print('OBS creds applied from env' if applied else 'OBS creds not set (using cluster session defaults)')

In [ ]:
# Scrape + enrich the next 12 months (heuristic scorer — no live LLM calls).
from special_days.agents import TurkeyAgent, InternationalAgent
from special_days.enrich import enrich, drop_long_events, DEFAULT_MAX_EVENT_SPAN_DAYS
from special_days.scoring import HeuristicScorer
from special_days.window import resolve_window
from special_days.models import SpecialDate

start, end = resolve_window(None, 12)  # today .. +12 months
records = []
for agent in (TurkeyAgent(), InternationalAgent()):
    records.extend(agent.collect(start, end, include_holidays=True, include_events=True))
records = list(dict.fromkeys(records))                 # de-dup overlap
records = drop_long_events(records, DEFAULT_MAX_EVENT_SPAN_DAYS)
records = enrich(records, scorer=HeuristicScorer())
records.sort(key=SpecialDate.sort_key)
print(f'{len(records)} special date(s) {start} -> {end}')

In [ ]:
# Write both Parquet datasets to OBS (full overwrite — idempotent at this volume).
LOCATION = 'obs://lakehouse-dev/special_events'
run_id = lakehouse.write(records, spark=spark, location=LOCATION)
print('run_id:', run_id)

In [ ]:
# Verify — read the datasets back by path.
raw = spark.read.parquet(f'{LOCATION}/special_days_raw')
feat = spark.read.parquet(f'{LOCATION}/special_days_features')
print('raw rows:', raw.count(), '| feature rows:', feat.count())
feat.orderBy('event_date').show(20, truncate=False)